# Game Data Manager
Manage the `games` MongoDB collection for soccer match data.

**Cells:**
1. Setup & imports
2. Init collection (indexes)
3. Drop collection
4. Push all data from Game_dataset
5. Inspect / stats

In [ ]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient, ASCENDING
from pymongo.errors import BulkWriteError
from dns import resolver
from tqdm import tqdm

# Load .env from project root
_nb_dir = Path(os.getcwd())
_project_root = _nb_dir.parent if _nb_dir.name == 'scripts' else _nb_dir
load_dotenv(_project_root / '.env', override=True)

MONGO_SRV        = os.getenv('MONGO_SRV')
SOCCER_DB_NAME   = os.getenv('SOCCER_DB_NAME', 'SoccerWikiDemo')
COLLECTION_NAME  = os.getenv('GAME_COLLECTION_NAME', 'games')
DATASET_DIR      = _project_root / 'app' / 'database' / 'Game_dataset'
CSV_PATH         = _project_root / 'app' / 'database' / 'game_database.csv'

assert MONGO_SRV, 'MONGO_SRV not set — check .env'

# Use public DNS to avoid corporate resolver issues
resolver.default_resolver = resolver.Resolver(configure=False)
resolver.default_resolver.nameservers = ['8.8.8.8', '1.1.1.1']

client     = MongoClient(MONGO_SRV)
db         = client[SOCCER_DB_NAME]
collection = db[COLLECTION_NAME]

print(f'✅ Connected to MongoDB: {SOCCER_DB_NAME}.{COLLECTION_NAME}')
print(f'✅ Dataset dir: {DATASET_DIR}')

In [ ]:
# ── Cell 2: Init collection (create indexes) ──────────────────────────────────
existing = db.list_collection_names()
if COLLECTION_NAME in existing:
    print(f'⚠️  Collection "{COLLECTION_NAME}" already exists. Creating indexes if missing...')
else:
    print(f'Creating collection "{COLLECTION_NAME}"...')

# Unique index on game_id
collection.create_index('game_id', unique=True, name='idx_game_id')

# Compound index for common search filters
collection.create_index(
    [('league', ASCENDING), ('season', ASCENDING), ('date', ASCENDING)],
    name='idx_league_season_date'
)
collection.create_index(
    [('home_team', ASCENDING), ('away_team', ASCENDING)],
    name='idx_teams'
)
collection.create_index('date', name='idx_date')

print(f'✅ Indexes created on "{COLLECTION_NAME}"')
for idx in collection.list_indexes():
    print(f'   {idx["name"]}: {idx["key"]}')

In [ ]:
# ── Cell 3: Drop collection ───────────────────────────────────────────────────
confirm = input(f'Type DROP to confirm dropping "{COLLECTION_NAME}": ')
if confirm.strip() == 'DROP':
    db.drop_collection(COLLECTION_NAME)
    print(f'🗑️  Collection "{COLLECTION_NAME}" dropped.')
else:
    print('Aborted.')

In [ ]:
# ── Cell 4: Push all data from Game_dataset ───────────────────────────────────
import pandas as pd

BATCH_SIZE = 100

# Read CSV as lookup for league/season/date/teams per file_path
df_csv = pd.read_csv(CSV_PATH)
csv_index = {row['file_path']: row.to_dict() for _, row in df_csv.iterrows()}
print(f'📄 CSV loaded: {len(csv_index)} entries')

def build_game_id(json_path: Path) -> str:
    """Use the match folder path relative to DATASET_DIR as game_id.
    e.g. england_epl_2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley
    """
    return str(json_path.parent.relative_to(DATASET_DIR)).replace('\\', '/')

def build_document(json_path: Path, csv_meta: dict) -> dict:
    """Build a MongoDB document from a JSON file + CSV metadata."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    game_id = build_game_id(json_path)

    doc = {
        'game_id' : game_id,
        'league'  : csv_meta.get('league', ''),
        'season'  : csv_meta.get('season', ''),
        'date'    : csv_meta.get('date', ''),
        'home_team': csv_meta.get('home_team', ''),
        'away_team': csv_meta.get('away_team', ''),
        'score'   : csv_meta.get('score', ''),
        'venue'   : csv_meta.get('venue', ''),
        'referee' : csv_meta.get('referee', ''),
        'raw'     : data,
    }
    return doc

# Walk Game_dataset and collect all JSON paths
all_json_paths = sorted(DATASET_DIR.rglob('*.json'))
print(f'📂 Found {len(all_json_paths)} JSON files')

inserted = 0
skipped  = 0
errors   = 0
batch    = []

for json_path in tqdm(all_json_paths, desc='Building documents'):
    try:
        rel = 'database/' + str(json_path.relative_to(DATASET_DIR.parent)).replace('\\', '/')
        csv_meta = csv_index.get(rel, {})
        doc = build_document(json_path, csv_meta)
        batch.append(doc)

        if len(batch) >= BATCH_SIZE:
            try:
                result = collection.insert_many(batch, ordered=False)
                inserted += len(result.inserted_ids)
            except BulkWriteError as e:
                inserted += e.details.get('nInserted', 0)
                skipped  += len([err for err in e.details.get('writeErrors', []) if err.get('code') == 11000])
            batch = []
    except Exception as e:
        errors += 1
        print(f'❌ Error on {json_path.name}: {e}')

# Flush remaining
if batch:
    try:
        result = collection.insert_many(batch, ordered=False)
        inserted += len(result.inserted_ids)
    except BulkWriteError as e:
        inserted += e.details.get('nInserted', 0)
        skipped  += len([err for err in e.details.get('writeErrors', []) if err.get('code') == 11000])

print(f'\n✅ Inserted: {inserted} | Skipped (duplicate): {skipped} | Errors: {errors}')
print(f'📊 Total documents in collection: {collection.count_documents({})}')

In [ ]:
# ── Cell 5: Inspect / stats ───────────────────────────────────────────────────
total = collection.count_documents({})
print(f'Total documents: {total}')

# Breakdown by league
pipeline = [{'$group': {'_id': '$league', 'count': {'$sum': 1}}}, {'$sort': {'_id': 1}}]
print('\nBy league:')
for r in collection.aggregate(pipeline):
    print(f"  {r['_id']}: {r['count']}")

# Sample document (without raw to keep output clean)
sample = collection.find_one({}, {'raw': 0})
print('\nSample document (no raw field):')
print(json.dumps(sample, indent=2, default=str))